# Лекция 5. Разбор упражнений

Упражнения из раздела 11 конспекта [`lecture05.md`](lecture05.md). Формат одинаковый: условие и его суть → разбор по пунктам → проверка кодом, по одной идее на ячейку.

На семинаре (за ноутбуком, [`seminar05.ipynb`](seminar05.ipynb)): 11.0 — разминка, 11.5 — BFGS руками, 11.7 — Гаусс–Ньютон. Дома: 11.1–11.4, 11.6, 11.8–11.10. Сначала попробуйте решить сами; краткие ответы без кода — [`exercises05.md`](exercises05.md).

В проверках используются методы и задачи из демо: `newton`, `newton_damped`, `bfgs`, `gauss_newton`, `gd`, функции Розенброка, `make_chain`, данные синуса. Их код — в файле [`l5helpers.py`](l5helpers.py), который подключает первая ячейка.

In [1]:
%matplotlib inline

from l5helpers import *  # палитра, методы, задачи, рисовальщики (см. файл рядом)

## 11.0. Разминка — счёт руками

> Шесть пунктов по минуте. (а) Один шаг Ньютона на $f=\tfrac12(x_1^2+9x_2^2)-x_1-9x_2$ из $(0,0)$. (б) $\sqrt2$ Ньютоном для $x^2-2$ из $x_0=1$. (в) Секущая для $f=x^4$ по точкам $1$ и $0.5$. (г) $J$, $J^\top J$ и $\nabla^2f$ для $r=(x_1-1,\ x_1x_2)$. (д) $\kappa$ Розенброка в $(1,1)$ по следу и определителю; почему Ньютону всё равно. (е) $H=\operatorname{diag}(1,-1)$: шаг Ньютона для $g=(1,0)$ и $g=(0,1)$ — спуск или подъём?
>
> *Суть: шаг Ньютона, секущая и структура МНК — на бумаге, без компьютера.*

**(а)** $\nabla f=(x_1-1,\ 9x_2-9)$, в нуле $g=(-1,-9)$; $H=\operatorname{diag}(1,9)$ постоянен. Шаг $p=-H^{-1}g=(1,\,1)$, новая точка $(1,1)$, и там $\nabla f=0$: функция квадратична, модель — это сама функция, одной итерации достаточно.

**(б)** $x_{k+1}=x_k-\dfrac{x_k^2-2}{2x_k}=\dfrac12\Big(x_k+\dfrac2{x_k}\Big)$: $x_1=1.5$, $x_2=1.41\overline{6}$, $x_3=1.414216$. Ошибки $0.086,\ 0.0025,\ 2.1\cdot10^{-6}$ — верных цифр $1$, $3$, $6$: удваиваются.

**(в)** $F=f'=4x^3$: $F(1)=4$, $F(0.5)=0.5$; $s=0.5-1=-0.5$, $y=0.5-4=-3.5$, оценка $f''\approx y/s=7$. Истинные $f''(0.5)=3$ и $f''(1)=12$: секущая даёт «среднюю» кривизну между точками (точно — $f''$ в некоторой промежуточной точке, здесь $12\xi^2=7$ при $\xi\approx0.76$).

**(г)** $J=\begin{pmatrix}1&0\\x_2&x_1\end{pmatrix}$, $J^\top J=\begin{pmatrix}1+x_2^2&x_1x_2\\x_1x_2&x_1^2\end{pmatrix}$. Вторая невязка $r_2=x_1x_2$ имеет гессиан $\begin{pmatrix}0&1\\1&0\end{pmatrix}$, первая — нулевой, так что отброшенный член $S=r_2\nabla^2r_2=x_1x_2\begin{pmatrix}0&1\\1&0\end{pmatrix}$ и $\nabla^2f=J^\top J+S$. Он равен нулю, когда $r_2=x_1x_2=0$ — в частности, в решении $(1,0)$, где обе невязки нулевые: там Гаусс–Ньютон совпадает с Ньютоном.

**(д)** $\operatorname{tr}=1002$, $\det=400$: $\lambda_{\max}\approx\operatorname{tr}=1001.6$, $\lambda_{\min}\approx\det/\operatorname{tr}=0.3994$, $\kappa\approx2508$. Ньютону это число безразлично из-за аффинной инвариантности (§2.4): замена координат, превращающая эллипсы в круги ($\kappa=1$), не меняет его путь, а $\kappa$ — свойство координат.

**(е)** $H^{-1}=\operatorname{diag}(1,-1)$. Для $g=(1,0)$: $p=(-1,0)$, $g^\top p=-1<0$ — спуск. Для $g=(0,1)$: $p=(0,1)$, $g^\top p=1>0$ — **подъём**: вдоль направления с отрицательной кривизной модель — перевёрнутая парабола, и её «стационарная точка» — максимум.

In [2]:
# (а): один шаг Ньютона на квадратичной попадает в минимум
Q0 = np.diag([1.0, 9.0]); c0 = np.array([-1.0, -9.0])
x = np.zeros(2); x = x - np.linalg.solve(Q0, Q0 @ x + c0)
print("(а) после одного шага:", x, "  grad =", Q0 @ x + c0)
assert np.allclose(x, [1, 1])

# (б): sqrt(2) — число верных цифр удваивается
x, errs = 1.0, []
for k in range(3):
    x = 0.5 * (x + 2 / x); errs.append(abs(x - np.sqrt(2)))
    print(f"(б) x_{k + 1} = {x:.7f}, ошибка {errs[-1]:.1e}")
assert np.allclose(errs, [0.0858, 0.00245, 2.1e-6], rtol=0.05)

# (в): секущая между 1 и 0.5 для f = x^4
F = lambda x: 4 * x ** 3
print("(в) y/s =", (F(0.5) - F(1)) / (0.5 - 1), " f''(0.5) =", 12 * 0.25, " f''(1) =", 12)

(а) после одного шага: [1. 1.]   grad = [0. 0.]
(б) x_1 = 1.5000000, ошибка 8.6e-02
(б) x_2 = 1.4166667, ошибка 2.5e-03
(б) x_3 = 1.4142157, ошибка 2.1e-06
(в) y/s = 7.0  f''(0.5) = 3.0  f''(1) = 12


In [3]:
# (г): J^T J + S равен конечно-разностному гессиану f = 1/2 ||r||^2
r_g = lambda x: np.array([x[0] - 1, x[0] * x[1]])
J_g = lambda x: np.array([[1.0, 0.0], [x[1], x[0]]])
f_g = lambda x: 0.5 * r_g(x) @ r_g(x)
x = np.array([0.7, 1.3]); h = 1e-5
H_fd = np.array([[(f_g(x + h * (ei + ej)) - f_g(x + h * (ei - ej)) - f_g(x - h * (ei - ej)) + f_g(x - h * (ei + ej))) / (4 * h * h)
                  for ej in np.eye(2)] for ei in np.eye(2)])
S = x[0] * x[1] * np.array([[0, 1], [1, 0]])
print("(г) J^T J + S =\n", (J_g(x).T @ J_g(x) + S).round(4), "\n    конечные разности =\n", H_fd.round(4))
assert np.allclose(J_g(x).T @ J_g(x) + S, H_fd, atol=1e-4)

# (д): след и определитель гессиана Розенброка в (1, 1)
H1 = rosen_hess(np.array([1.0, 1.0])); tr, det = np.trace(H1), np.linalg.det(H1)
print(f"(д) tr = {tr:.0f}, det = {det:.0f}: lambda ~ {tr:.1f} и {det / tr:.4f}, kappa ~ {tr ** 2 / det:.0f}; точно: {np.linalg.eigvalsh(H1).round(4)}")

# (е): знак g^T p при индефинитном H
H = np.diag([1.0, -1.0])
for g in (np.array([1.0, 0.0]), np.array([0.0, 1.0])):
    p = -np.linalg.solve(H, g); print(f"(е) g = {g}: p = {p}, g^T p = {g @ p:+.0f} -> {'спуск' if g @ p < 0 else 'подъём'}")

(г) J^T J + S =
 [[2.69 1.82]
 [1.82 0.49]] 
    конечные разности =
 [[2.69 1.82]
 [1.82 0.49]]
(д) tr = 1002, det = 400: lambda ~ 1002.0 и 0.3992, kappa ~ 2510; точно: [3.9940000e-01 1.0016006e+03]
(е) g = [1. 0.]: p = [-1.  0.], g^T p = -1 -> спуск
(е) g = [0. 1.]: p = [-0.  1.], g^T p = +1 -> подъём


## 11.1. Два прочтения шага Ньютона

> (а) $\nabla m_k(p)=0$ и линеаризация $\nabla f(x_k+p)\approx0$ дают один $p$. (б) Аффинная инвариантность: при $x=Au$ шаги связаны как $A\tilde p=p$. (в) У градиентного спуска инвариантности нет: $f=\tfrac12(x_1^2+25x_2^2)$, замена $u_2=5x_2$.
>
> *Суть: минимум модели и нуль линеаризованного градиента — одна формула; координаты она не видит.*

**(а)** $m_k(p)=f_k+g_k^\top p+\tfrac12p^\top H_kp$, $\nabla m_k(p)=g_k+H_kp$. Линеаризация: $\nabla f(x_k+p)\approx\nabla f(x_k)+\nabla^2f(x_k)p=g_k+H_kp$. Оба нуля — решение одной системы $H_kp=-g_k$: градиент модели *и есть* линеаризованный градиент функции (модель — квадратичная, её градиент — линейный).

**(б)** $\tilde f(u)=f(Au)$. Цепное правило: $\nabla\tilde f(u)=A^\top\nabla f(Au)$, $\nabla^2\tilde f(u)=A^\top\nabla^2f(Au)A$. Шаг в $u$: $\tilde p=-(A^\top HA)^{-1}A^\top g=-A^{-1}H^{-1}A^{-\top}A^\top g=-A^{-1}H^{-1}g=A^{-1}p$. Значит $A\tilde p=p$: сдвиг в исходных координатах тот же, что бы ни взять за $A$. По индукции совпадают все итерации.

**(в)** $x=Au$ с $A=\operatorname{diag}(1,\tfrac15)$ (тогда $u_2=5x_2$). Шаг спуска в $u$: $-\alpha\nabla\tilde f=-\alpha A^\top g$; в координатах $x$ это $A(-\alpha A^\top g)=-\alpha AA^\top g=-\alpha\operatorname{diag}(1,\tfrac1{25})\,g\ne-\alpha g$. Спуск в новых координатах — это спуск с *другой* матрицей масштабирования, то есть другой метод (предобуславливание, Л4 §5.4); в $u$ эллипсы стали кругами, и он сходится за один шаг, а в $x$ — за $378$.

In [4]:
# (б): путь Ньютона в координатах u, пересчитанный в x, совпадает с путём в x
A = np.array([[2.0, 0.5], [0.0, 0.25]])          # произвольная невырожденная замена x = A u
f_u = lambda u: rosen(A @ u)
grad_u = lambda u: A.T @ rosen_grad(A @ u)
hess_u = lambda u: A.T @ rosen_hess(A @ u) @ A
_, k_x, path_x = newton(rosen_grad, rosen_hess, X0_ROSEN)
_, k_u, path_u = newton(grad_u, hess_u, np.linalg.solve(A, X0_ROSEN))
print(f"(б) итераций в x: {k_x}, в u: {k_u};  max |A u_k - x_k| = {np.abs(path_u @ A.T - path_x).max():.1e}")
assert k_x == k_u and np.allclose(path_u @ A.T, path_x, atol=1e-6)

# (в): спуск в u и в x — разные пути
Qx = np.diag([1.0, 25.0]); Au = np.diag([1.0, 0.2])
_, kx, px = gd(lambda x: 0.5 * x @ Qx @ x, lambda x: Qx @ x, [5.0, 1.0], alpha=1 / 25)
_, ku, pu = gd(lambda u: 0.5 * u @ u, lambda u: u, [5.0, 5.0], alpha=1.0)
print(f"(в) спуск в x: {kx} итераций; в u (после замены u2 = 5 x2): {ku}; второй шаг в x: {px[1].round(3)}, пересчитанный из u: {(Au @ pu[1]).round(3)}")
assert kx > 100 and ku == 1

(б) итераций в x: 7, в u: 7;  max |A u_k - x_k| = 3.5e-12
(в) спуск в x: 378 итераций; в u (после замены u2 = 5 x2): 1; второй шаг в x: [4.8 0. ], пересчитанный из u: [0. 0.]


## 11.2. Квадратичная сходимость (домашняя)

> (а) Доказательство §3.1 подробно. (б) Оценить $M$ и $m$ для Розенброка около $(1,1)$ и радиус гарантированной сходимости. (в) Сравнить $C=M/2m$ с измеренными $C_k$.
>
> *Суть: ошибка Ньютона — это ошибка модели, а она квадратична.*

**(а)** Пусть $e_k=x_k-x^\ast$, $H_k=\nabla^2f(x_k)$, $g_k=\nabla f(x_k)$. Шаг: $x_{k+1}-x^\ast=e_k-H_k^{-1}g_k=H_k^{-1}(H_ke_k-g_k)$. Поскольку $\nabla f(x^\ast)=0$, $g_k=\int_0^1\nabla^2f(x^\ast+te_k)\,e_k\,dt$ (формула Ньютона–Лейбница для $\nabla f$ вдоль отрезка). Тогда $H_ke_k-g_k=\int_0^1\big(\nabla^2f(x_k)-\nabla^2f(x^\ast+te_k)\big)e_k\,dt$. **Липшицевость** гессиана: $\lVert\nabla^2f(x_k)-\nabla^2f(x^\ast+te_k)\rVert\le M\lVert x_k-x^\ast-te_k\rVert=M(1-t)\lVert e_k\rVert$. Интегрируем: $\lVert H_ke_k-g_k\rVert\le M\lVert e_k\rVert^2\int_0^1(1-t)dt=\tfrac M2\lVert e_k\rVert^2$. **Условие $\nabla^2f\succeq mI$** даёт $\lVert H_k^{-1}\rVert\le1/m$. Итого $\lVert e_{k+1}\rVert\le\frac M{2m}\lVert e_k\rVert^2$. Если $\lVert e_0\rVert<2m/M$, то $\lVert e_1\rVert<\lVert e_0\rVert$, итерации остаются в окрестности, и по индукции $\frac M{2m}\lVert e_k\rVert\le\big(\frac M{2m}\lVert e_0\rVert\big)^{2^k}\to0$.

**(б)** Гессиан Розенброка зависит от $x$ линейно по $x_2$ и квадратично по $x_1$: $\partial H/\partial x_1=\begin{pmatrix}2400x_1&-400\\-400&0\end{pmatrix}$, $\partial H/\partial x_2=\begin{pmatrix}-400&0\\0&0\end{pmatrix}$. Около $(1,1)$ нормы этих матриц $\approx2465$ и $400$; грубая оценка $\sqrt{2465^2+400^2}\approx2500$, а честный замер $\max\lVert H(x)-H(y)\rVert/\lVert x-y\rVert$ по случайным парам в шаре радиуса $0.3$ даёт $M\approx3100$ (направления, где обе производные складываются). $m=\lambda_{\min}(\nabla^2f(1,1))=0.3994$. $C=M/2m\approx3900$; теорема гарантирует сходимость из шара радиуса $1/C\approx3\cdot10^{-4}$.

**(в)** Измеренные $C_k$ — $0.003$–$0.86$, в тысячи раз меньше $C$. Оценка пессимистична по трём причинам: $M$ берётся как худшая по всем направлениям константа, а ошибка идёт вдоль долины; $1/m$ — худшая норма $H^{-1}$, а реальный вектор $H_ke_k-g_k$ не обязан лежать вдоль самого пологого направления; наконец, теорема описывает *локальный тип* сходимости, а не область: метод сошёлся из $\lVert e_0\rVert=2.2$, в $7000$ раз дальше гарантированного радиуса.

In [5]:
# (б)-(в): оценка M и m около (1, 1) против измеренных C_k
dH1 = np.array([[2400.0, -400.0], [-400.0, 0.0]]); dH2 = np.array([[-400.0, 0.0], [0.0, 0.0]])
M_rough = np.hypot(np.linalg.norm(dH1, 2), np.linalg.norm(dH2, 2))
rng = np.random.default_rng(2026); M = 0.0                     # замер: наибольшее ||H(x)-H(y)|| / ||x-y|| в шаре радиуса 0.3 вокруг (1,1)
for _ in range(5000):
    x = 1 + 0.3 * rng.uniform(-1, 1, 2); y = 1 + 0.3 * rng.uniform(-1, 1, 2)
    M = max(M, np.linalg.norm(rosen_hess(x) - rosen_hess(y), 2) / np.linalg.norm(x - y))
m = np.linalg.eigvalsh(rosen_hess(np.array([1.0, 1.0])))[0]
C = M / (2 * m)
print(f"грубая оценка M ~ {M_rough:.0f}; замер по 5000 случайным парам: M ~ {M:.0f}; m = {m:.4f}; C = M/2m ~ {C:.0f}; радиус гарантии 1/C ~ {1 / C:.1e}")
_, k, path = newton(rosen_grad, rosen_hess, X0_ROSEN)
e = np.linalg.norm(path - 1, axis=1)
Ck = e[1:-1] / e[:-2] ** 2
print("измеренные C_k:", np.array2string(Ck, precision=3), " — все меньше C:", np.all(Ck < C), f"; метод сошёлся из |e_0| = {e[0]:.1f}, что в {e[0] * C:.0f} раз дальше радиуса гарантии")
assert np.all(Ck < C) and 2500 < M < 3500

грубая оценка M ~ 2497; замер по 5000 случайным парам: M ~ 3087; m = 0.3994; C = M/2m ~ 3865; радиус гарантии 1/C ~ 2.6e-04
измеренные C_k: [0.456 0.857 0.027 0.243 0.003 0.2  ]  — все меньше C: True ; метод сошёлся из |e_0| = 2.2, что в 8503 раз дальше радиуса гарантии


## 11.3. Сломай метод

> (а) Порог старта для $\ln\cosh x$ (2-цикл $x_1=-x_0$), $\approx1.0886$. (б) То же для $\arctan x=0$, $\approx1.3917$. (в) Объяснить по картинке. (г) Backtracking по Армихо: сходится ли из $x_0=5$?
>
> *Суть: у Ньютона есть порог старта, за которым он расходится; демпфирование снимает порог.*

**(а)** Итерация $x_{k+1}=x_k-\tanh x_k\cosh^2x_k=x_k-\tfrac12\sinh2x_k$. 2-цикл: $x_1=-x_0\iff2x_0=\tfrac12\sinh2x_0$. Функция $\tfrac12\sinh2x-2x$ отрицательна при малых $x$ и растёт экспоненциально, единственный положительный корень $x_0^\ast\approx1.0886$. При $|x_0|<x_0^\ast$ выполняется $|x_1|<|x_0|$ (шаг «недолетает» до $-x_0$), ошибки убывают, и дальше квадратичная сходимость; при $|x_0|>x_0^\ast$ — $|x_1|>|x_0|$, и каждая следующая точка дальше предыдущей.

**(б)** Для $F=\arctan$: $x_{k+1}=x_k-\arctan(x_k)(1+x_k^2)$; порог — корень $2x=\arctan(x)(1+x^2)$, $x^\ast\approx1.3917$.

**(в)** Кривизна $f''=1/\cosh^2x$ убывает с ростом $|x|$: чем дальше старт, тем шире парабола-модель и тем дальше её вершина от нуля. Ровно на пороге вершина попадает в $-x_0$ — зеркальную точку, и картина повторяется с другим знаком.

**(г)** Направление Ньютона $p=-f'/f''$ в одномерном случае — всегда спуск (при $f''>0$). Backtracking делит шаг пополам, пока $f$ не уменьшится достаточно, поэтому «перелёт» невозможен: метод сходится из любого старта. Из $x_0=5$ полный шаг равен $-\tfrac12\sinh10\approx-5500$, Армихо принимает его после $\approx13$ делений; затем ещё несколько итераций с $\alpha<1$, и как только $|x|<1.09$, шаги принимаются целиком — квадратичный финал. Всего около десяти итераций.

In [6]:
from scipy.optimize import brentq
thr_lncosh = brentq(lambda x: 0.5 * np.sinh(2 * x) - 2 * x, 0.5, 2.0)
thr_atan = brentq(lambda x: np.arctan(x) * (1 + x ** 2) - 2 * x, 0.5, 2.0)
print(f"(а) порог для ln cosh: {thr_lncosh:.4f}   (б) порог для arctan: {thr_atan:.4f}")
assert abs(thr_lncosh - 1.0886) < 1e-3 and abs(thr_atan - 1.3917) < 1e-3
for x0 in (thr_lncosh - 0.01, thr_lncosh, thr_lncosh + 0.01):
    x = x0; seq = [x]
    for _ in range(5): x = x - 0.5 * np.sinh(2 * x); seq.append(x)
    print(f"    ln cosh из {x0:.4f}: {np.array2string(np.array(seq), precision=3)}")

(а) порог для ln cosh: 1.0887   (б) порог для arctan: 1.3917
    ln cosh из 1.0787: [ 1.079 -1.054  0.975 -0.746  0.31  -0.02 ]
    ln cosh из 1.0887: [ 1.089 -1.089  1.089 -1.089  1.089 -1.089]
    ln cosh из 1.0987: [ 1.099e+00 -1.124e+00  1.216e+00 -1.607e+00  4.605e+00 -2.494e+03]


In [7]:
# (г): демпфированный Ньютон на ln cosh из x0 = 5
f1 = lambda x: np.log(np.cosh(x[0])); g1 = lambda x: np.array([np.tanh(x[0])]); h1 = lambda x: np.array([[1 / np.cosh(x[0]) ** 2]])
x_d, k_d, path_d = newton_damped(f1, g1, h1, [5.0], tol=1e-10)
x_pure = [5.0]
for _ in range(2):                                       # чистый Ньютон: два шага, дальше переполнение
    x_pure.append(x_pure[-1] - 0.5 * np.sinh(2 * x_pure[-1]))
print(f"(г) демпфированный Ньютон из 5: {k_d} итераций, путь {np.array2string(path_d.ravel(), precision=3)}")
print(f"    чистый Ньютон из 5: {x_pure[0]} -> {x_pure[1]:.0f} -> {x_pure[2]:.1e} — расходится")
assert abs(x_d[0]) < 1e-8 and k_d < 20

(г) демпфированный Ньютон из 5: 4 итераций, путь [ 5.000e+00 -3.776e-01  3.692e-02 -3.355e-05  2.518e-14]
    чистый Ньютон из 5: 5.0 -> -5502 -> inf — расходится


/tmp/ipykernel_1781/1665587353.py:2: RuntimeWarning: overflow encountered in cosh
  f1 = lambda x: np.log(np.cosh(x[0])); g1 = lambda x: np.array([np.tanh(x[0])]); h1 = lambda x: np.array([[1 / np.cosh(x[0]) ** 2]])
/tmp/ipykernel_1781/1665587353.py:6: RuntimeWarning: overflow encountered in sinh
  x_pure.append(x_pure[-1] - 0.5 * np.sinh(2 * x_pure[-1]))


## 11.4. Ньютон на седловой модели и демпфирование

> (а) $f=x_1^2-x_2^2+x_2^4/4$: стационарные точки, их тип; из любой точки с $x_2^2<1/3$ шаг Ньютона ведёт в седло $(0,0)$. (б) Розенброк: $\det\nabla^2f<0\iff x_2>x_1^2+0.005$; точка с $g^\top p>0$. (в) Почему демпфированный Ньютон вблизи минимума принимает $\alpha=1$.
>
> *Суть: при индефинитном гессиане «минимум модели» — седло, и шаг может вести вверх.*

**(а)** $\nabla f=(2x_1,\ -2x_2+x_2^3)$: нули при $x_1=0$ и $x_2\in\{0,\pm\sqrt2\}$. Гессиан $\operatorname{diag}(2,\ -2+3x_2^2)$: в $(0,\pm\sqrt2)$ — $\operatorname{diag}(2,4)\succ0$, минимумы ($f=-1$); в $(0,0)$ — $\operatorname{diag}(2,-2)$, **седло**. Шаг Ньютона покоординатный: $x_1\to0$ сразу, а $x_2\to x_2-\dfrac{x_2^3-2x_2}{3x_2^2-2}=\dfrac{2x_2^3}{3x_2^2-2}$. При $x_2^2<2/3$ знаменатель отрицателен, и $|x_{2,\mathrm{new}}|=\dfrac{2|x_2|^3}{2-3x_2^2}<|x_2|\iff x_2^2<2/5$. Значит при $x_2^2<1/3$ (и даже $<2/5$) $|x_2|$ убывает, причём кубически — метод сходится к седлу $(0,0)$, а не к одному из минимумов: модель там седловая, и её стационарная точка — перевал.

**(б)** $\det\nabla^2f=200\,(2-400x_2+1200x_1^2)-160\,000x_1^2=80\,000\,(x_1^2-x_2+0.005)$ — отрицателен ровно над параболой. В точке $(-1.06,\ 1.18)$ (чуть выше дна долины): $\det<0$ и $g^\top p>0$ — шаг Ньютона идёт вверх по $f$. В большинстве других точек над параболой $g^\top p<0$: модель седловая, но её седло лежит ниже текущей точки.

**(в)** Вблизи минимума модель точна с ошибкой $O(\lVert p\rVert^3)$, поэтому $f(x+p)=f(x)+g^\top p+\tfrac12p^\top Hp+O(\lVert p\rVert^3)=f(x)+\tfrac12g^\top p+O(\lVert p\rVert^3)$ (так как $Hp=-g$). Условие Армихо требует $f(x+p)\le f(x)+c\,g^\top p$; при $c<\tfrac12$ и малом $\lVert p\rVert$ оно выполнено для полного шага, потому что $\tfrac12g^\top p<c\,g^\top p$ (оба отрицательны). Принимается $\alpha=1$, и метод становится чистым Ньютоном — квадратичная сходимость сохраняется.

In [8]:
f4 = lambda x: x[0] ** 2 - x[1] ** 2 + x[1] ** 4 / 4
g4 = lambda x: np.array([2 * x[0], -2 * x[1] + x[1] ** 3])
h4 = lambda x: np.array([[2.0, 0.0], [0.0, -2 + 3 * x[1] ** 2]])
for x0 in ([0.5, 0.5], [1.0, -0.3], [0.2, 1.0]):
    x, k, path = newton(g4, h4, x0)
    lam = np.linalg.eigvalsh(h4(x))
    print(f"(а) чистый Ньютон из {x0}: пришёл в {x.round(4)} за {k} итераций -> {'седло' if lam.min() < 0 else 'минимум'}")
x, k, _ = newton(g4, h4, [0.5, 0.5]); assert np.allclose(x, 0, atol=1e-8)
x, k, _ = newton_damped(f4, g4, h4, [0.5, 0.5])
print(f"    демпфированный Ньютон из (0.5, 0.5): пришёл в {x.round(4)} за {k} итераций (регуляризация H + tau I при индефинитном H)")

(а) чистый Ньютон из [0.5, 0.5]: пришёл в [0. 0.] за 4 итераций -> седло
(а) чистый Ньютон из [1.0, -0.3]: пришёл в [0. 0.] за 3 итераций -> седло
(а) чистый Ньютон из [0.2, 1.0]: пришёл в [0.     1.4142] за 6 итераций -> минимум
    демпфированный Ньютон из (0.5, 0.5): пришёл в [0.     1.4142] за 5 итераций (регуляризация H + tau I при индефинитном H)


In [9]:
# (б): точка над дном долины, где шаг Ньютона идёт вверх
x = np.array([-1.06, 1.18]); g = rosen_grad(x); H = rosen_hess(x); p = -np.linalg.solve(H, g)
print(f"(б) det H = {np.linalg.det(H):.0f}, формула 80000 (x1^2 - x2 + 0.005) = {80000 * (x[0] ** 2 - x[1] + 0.005):.0f};  g^T p = {g @ p:.3f} > 0;  f(x + 0.01 p) - f(x) = {rosen(x + 0.01 * p) - rosen(x):+.4f}")
assert g @ p > 0 and rosen(x + 0.01 * p) > rosen(x)

# (в): у демпфированного Ньютона на Розенброке последние шаги приняты с alpha = 1
x_d, k_d, path_d = newton_damped(rosen, rosen_grad, rosen_hess, X0_ROSEN)
full = [np.allclose(path_d[k + 1] - path_d[k], -np.linalg.solve(rosen_hess(path_d[k]), rosen_grad(path_d[k]))) for k in range(len(path_d) - 1)]
print("(в) принят полный шаг (alpha = 1) на итерациях:", np.array(full).astype(int), " — с некоторого момента всегда")
assert all(full[-6:])

(б) det H = -4112, формула 80000 (x1^2 - x2 + 0.005) = -4112;  g^T p = 0.189 > 0;  f(x + 0.01 p) - f(x) = +0.0019
(в) принят полный шаг (alpha = 1) на итерациях: [1 0 1 1 1 0 1 1 1 0 1 1 1 0 1 1 1 1 1 1 1 1]  — с некоторого момента всегда


## 11.5. BFGS руками

> $f=\tfrac12(x_1^2+50x_2^2)$, $x_0=(50,1)$, $H_0=I$, точный линейный поиск. (а) Первый шаг, $s_0$, $y_0$, $s_0^\top y_0>0$. (б) $H_1$ и проверка $H_1y_0=s_0$. (в) Второй шаг, $H_2=Q^{-1}$, $x_2=0$.
>
> *Суть: два обновления на квадратичной $2\times2$ восстанавливают $Q^{-1}$ целиком.* Разбирается на семинаре.

**(а)** $g_0=Qx_0=(50,50)$, $p_0=-g_0$. Точный шаг $\alpha_0=\dfrac{g_0^\top g_0}{g_0^\top Qg_0}=\dfrac{5000}{2500+125\,000}=\dfrac{2}{51}\approx0.0392$. $s_0=\alpha_0p_0=-\tfrac{100}{51}(1,1)\approx(-1.961,-1.961)$, $x_1=(48.04,\,-0.961)$, $y_0=Qs_0=-\tfrac{100}{51}(1,50)$, $s_0^\top y_0=\tfrac{100^2}{51^2}\cdot51=\tfrac{10\,000}{51}\approx196>0$ — положительная кривизна вдоль шага, как и должно быть на выпуклой функции.

**(б)** По формуле $H_1=(I-\rho sy^\top)H_0(I-\rho ys^\top)+\rho ss^\top$ с $\rho=1/s^\top y$. Подстановка даёт $H_1\approx\begin{pmatrix}1.942&-0.019\\-0.019&0.0204\end{pmatrix}$ — проверяем в коде, что $H_1y_0=s_0$ (секущее уравнение) и $H_1\succ0$. Обратите внимание: вдоль направления $s_0$ модель уже знает кривизну, а в поперечном направлении $H_1$ ещё «помнит» единицу.

**(в)** $g_1=Qx_1=(48.04,\,-48.04)$, $p_1=-H_1g_1$, точный шаг, $x_2=0$ (с точностью округления) и $H_2=\operatorname{diag}(1,\,0.02)=Q^{-1}$. На квадратичной функции $n$ линейно независимых пар $(s,y)$ с точным поиском полностью определяют $Q^{-1}$: секущее уравнение $H_{k+1}y_j=s_j$ при точном поиске сохраняется для всех предыдущих $j$, и после $n$ шагов $H_n=Q^{-1}$.

In [10]:
Q = np.diag([1.0, 50.0]); x = np.array([50.0, 1.0]); H = np.eye(2)
for k in range(2):
    g = Q @ x; p = -H @ g; a = -(g @ p) / (p @ Q @ p)
    s = a * p; x_new = x + s; y = Q @ x_new - g; rho = 1 / (s @ y); I = np.eye(2)
    print(f"шаг {k}: alpha = {a:.4f}, s = {s.round(4)}, y = {y.round(4)}, s^T y = {s @ y:.3f}")
    H = (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) + rho * np.outer(s, s)
    print(f"        H_{k + 1} = {np.array2string(H, precision=4, suppress_small=True).replace(chr(10), ' ')},  H y = s: {np.allclose(H @ y, s)},  H > 0: {np.all(np.linalg.eigvalsh(H) > 0)}")
    x = x_new
print("x_2 =", x.round(10), "  H_2 == Q^{-1}:", np.allclose(H, np.linalg.inv(Q)))
assert np.allclose(x, 0, atol=1e-9) and np.allclose(H, np.linalg.inv(Q))
# то же самое функцией bfgs из l5helpers
x_b, k_b, _, _ = bfgs(lambda x: 0.5 * x @ Q @ x, lambda x: Q @ x, [50.0, 1.0], tol=1e-10, Q=Q)
print("bfgs(..., Q=Q):", k_b, "итерации,  x =", x_b.round(10))

шаг 0: alpha = 0.0392, s = [-1.9608 -1.9608], y = [ -1.9608 -98.0392], s^T y = 196.078
        H_1 = [[ 1.9419 -0.0188]  [-0.0188  0.0204]],  H y = s: True,  H > 0: True
шаг 1: alpha = 0.5100, s = [-48.0392   0.9608], y = [-48.0392  48.0392], s^T y = 2353.922
        H_2 = [[1.   0.  ]  [0.   0.02]],  H y = s: True,  H > 0: True
x_2 = [-0.  0.]   H_2 == Q^{-1}: True
bfgs(..., Q=Q): 2 итерации,  x = [-0.  0.]


## 11.6. Свойства обновления BFGS (домашняя)

> (а) $H_{k+1}y_k=s_k$ подстановкой. (б) $H_k\succ0$, $s_k^\top y_k>0\Rightarrow H_{k+1}\succ0$. (в) Почему на выпуклой функции $s^\top y>0$ всегда, а на невыпуклой — нет; пример $f=-x^2$.
>
> *Суть: симметрия, секущее уравнение и положительная определённость — всё проверяется подстановкой.*

**(а)** $(I-\rho ys^\top)y=y-\rho y(s^\top y)=y-y=0$, поэтому первое слагаемое $(I-\rho sy^\top)H(I-\rho ys^\top)y$ обращается в нуль, а второе даёт $\rho s(s^\top y)=s$. Итого $H_{k+1}y=s$. Симметрия видна из формы записи: $A H A^\top+\rho ss^\top$ с $A=I-\rho sy^\top$.

**(б)** Для $z\ne0$: $z^\top H_{k+1}z=w^\top H_kw+\rho(s^\top z)^2$, где $w=(I-\rho ys^\top)z=z-\rho(s^\top z)y$. Оба слагаемых $\ge0$ ($H_k\succ0$, $\rho>0$). Сумма равна нулю, только если $w=0$ и $s^\top z=0$; но тогда $w=z-0=z=0$ — противоречие. Значит $z^\top H_{k+1}z>0$: положительная определённость сохраняется, и $p_{k+1}=-H_{k+1}g_{k+1}$ — направление спуска.

**(в)** $s^\top y=(x_{k+1}-x_k)^\top(\nabla f(x_{k+1})-\nabla f(x_k))$ — это монотонность градиента, равносильная выпуклости (Л2, §4): для выпуклых $f$ она неотрицательна всегда (строго — для строго выпуклых). Для $f=-x^2$: $\nabla f=-2x$, $y=-2s$, $s^\top y=-2s^2<0$ при любом шаге — обновление BFGS дало бы матрицу, не положительно определённую. Поэтому на невыпуклых функциях нужен линейный поиск, который обеспечивает $s^\top y>0$ (условие кривизны Вольфе, лекция 6), либо пропуск обновления, как в нашем коде.

In [11]:
rng = np.random.default_rng(2026)
ok = True
for _ in range(200):
    n = 4
    A = rng.standard_normal((n, n)); H = A @ A.T + 0.1 * np.eye(n)       # случайная H > 0
    s = rng.standard_normal(n); y = rng.standard_normal(n)
    if s @ y <= 0: y = -y                                                # обеспечиваем s^T y > 0
    rho = 1 / (s @ y); I = np.eye(n)
    Hn = (I - rho * np.outer(s, y)) @ H @ (I - rho * np.outer(y, s)) + rho * np.outer(s, s)
    ok &= np.allclose(Hn @ y, s) and np.allclose(Hn, Hn.T) and np.all(np.linalg.eigvalsh(Hn) > 0)
print("(а)-(б) 200 случайных обновлений: секущее уравнение, симметрия и H > 0 выполнены:", ok)
assert ok
# (в): на f = -x^2 условие кривизны нарушено
s_ = 0.3; y_ = -2 * s_
print(f"(в) f = -x^2: s = {s_}, y = {y_}, s^T y = {s_ * y_} < 0 — обновлять нельзя")

(а)-(б) 200 случайных обновлений: секущее уравнение, симметрия и H > 0 выполнены: True
(в) f = -x^2: s = 0.3, y = -0.6, s^T y = -0.18 < 0 — обновлять нельзя


## 11.7. Гаусс–Ньютон

> (а) $\nabla f=J^\top r$, $\nabla^2f=J^\top J+\sum r_i\nabla^2r_i$. (б) Шаг Гаусса–Ньютона — решение $\min_p\lVert Jp+r\rVert^2$; при полном ранге $J$ — направление спуска. (в) Столбцы $J$ для синуса; что при $x_1=0$; почему Ньютон туда пришёл, а Гаусс–Ньютон — нет. (г) Для $r=Ax-b$ один шаг даёт решение линейного МНК.
>
> *Суть: половина гессиана бесплатна, всегда $\succeq0$ и даёт спуск; вторая половина вдали от решения вредна.* Разбирается на семинаре.

**(а)** $f=\tfrac12\sum_ir_i^2$. $\partial f/\partial x_j=\sum_ir_i\,\partial r_i/\partial x_j=(J^\top r)_j$. Дифференцируем ещё раз: $\partial^2f/\partial x_j\partial x_l=\sum_i\big(\partial r_i/\partial x_j\cdot\partial r_i/\partial x_l+r_i\,\partial^2r_i/\partial x_j\partial x_l\big)=(J^\top J)_{jl}+\sum_ir_i(\nabla^2r_i)_{jl}$.

**(б)** Нормальные уравнения для $\min_p\lVert Jp+r\rVert^2$: $J^\top(Jp+r)=0\iff J^\top Jp=-J^\top r$ — это и есть система Гаусса–Ньютона. Направление спуска: $g^\top p=r^\top J\,p=-r^\top J(J^\top J)^{-1}J^\top r=-\lVert(J^\top J)^{-1/2}J^\top r\rVert^2<0$, если $J^\top r\ne0$ и $J^\top J\succ0$ (полный столбцовый ранг).

**(в)** $r_i=x_1\sin(x_2t_i+x_3)-y_i$; столбцы $J$: $\sin(x_2t_i+x_3)$, $x_1t_i\cos(x_2t_i+x_3)$, $x_1\cos(x_2t_i+x_3)$. При $x_1=0$ второй и третий столбцы — нули: $J$ ранга 1, $J^\top J$ вырождена. Градиент $J^\top r$ при $x_1=0$ равен $(\sum_i\sin(x_2t_i+x_3)(-y_i),\ 0,\ 0)$ — обращается в нуль, когда синус с данными частотой и фазой ортогонален данным; таких $(x_2,x_3)$ — целые кривые. Это стационарные точки с $f=\tfrac12\lVert y\rVert^2$, никакого отношения к подгонке не имеющие. Ньютон из $(1,3,0)$ использует полный гессиан, в котором вдали от решения член $S=\sum r_i\nabla^2r_i$ велик и делает модель седловой: шаги ведут не к минимуму, а к ближайшему нулю градиента, и за 9 итераций амплитуда гаснет. У Гаусса–Ньютона $J^\top J\succeq0$, шаг — всегда спуск по $f$, а в точках с $x_1=0$ значение $f$ *больше*, чем в старте, — туда он прийти не может.

**(г)** $J=A$, $r=Ax-b$: шаг $p$ решает $\min\lVert A(x+p)-b\rVert^2$, то есть $x+p=\arg\min\lVert Ax'-b\rVert$ — решение линейного МНК, за одну итерацию из любого старта (как Ньютон на квадратичной функции).

In [12]:
t, y, starts = sine_data()
resid, jac, f_s, grad_s, hess_s = sine_problem(t, y)
# (а): формулы против конечных разностей — проверено в демо (0); здесь — что S = hess - J^T J действительно пропорционален невязкам
x = np.array([1.5, 2.8, 0.4]); S = hess_s(x) - jac(x).T @ jac(x)
x_small = np.array([2.1014, 2.9967, 0.7003])      # почти решение: невязки малы
print(f"(а) ||S|| вдали от решения: {np.linalg.norm(S):.1f};  в решении: {np.linalg.norm(hess_s(x_small) - jac(x_small).T @ jac(x_small)):.1f};  ||J^T J|| там: {np.linalg.norm(jac(x_small).T @ jac(x_small)):.0f}")

# (б): шаг Гаусса–Ньютона = lstsq и всегда спуск
for x in (X0_SINE, np.array([-2.0, 1.0, 1.0]), np.array([0.5, 4.0, -1.0])):
    p_lstsq = np.linalg.lstsq(jac(x), -resid(x), rcond=None)[0]
    p_normal = np.linalg.solve(jac(x).T @ jac(x), -jac(x).T @ resid(x))
    print(f"(б) x = {x}: lstsq и нормальные уравнения совпадают: {np.allclose(p_lstsq, p_normal)},  g^T p = {grad_s(x) @ p_lstsq:.2f} < 0")

(а) ||S|| вдали от решения: 461.5;  в решении: 8.3;  ||J^T J|| там: 1936
(б) x = [1. 3. 0.]: lstsq и нормальные уравнения совпадают: True,  g^T p = -65.46 < 0
(б) x = [-2.  1.  1.]: lstsq и нормальные уравнения совпадают: True,  g^T p = -79.00 < 0
(б) x = [ 0.5  4.  -1. ]: lstsq и нормальные уравнения совпадают: True,  g^T p = -6.20 < 0


In [13]:
# (в): куда приходят Ньютон и Гаусс–Ньютон; ранг J при x1 = 0
x_n, k_n, _ = newton(grad_s, hess_s, X0_SINE, tol=1e-6)
x_g, k_g, _ = gauss_newton(resid, jac, X0_SINE)
print(f"(в) Ньютон: {x_n.round(4)} за {k_n} итераций, f = {f_s(x_n):.2f};  Гаусс–Ньютон: {x_g.round(4)} за {k_g}, f = {f_s(x_g):.4f}")
print(f"    ранг J в точке Ньютона: {np.linalg.matrix_rank(jac(x_n))},  |grad f| там: {np.linalg.norm(grad_s(x_n)):.1e},  f в старте: {f_s(X0_SINE):.2f} < f в точке Ньютона: {f_s(x_n):.2f}")
assert abs(x_n[0]) < 1e-6 and k_g == 7

# (г): линейные невязки — один шаг
rng = np.random.default_rng(2026); A = rng.standard_normal((30, 4)); b = rng.standard_normal(30)
x_lin, k_lin, _ = gauss_newton(lambda x: A @ x - b, lambda x: A, np.zeros(4), tol=1e-10)
print(f"(г) линейный МНК: Гаусс–Ньютон за {k_lin} итерацию, |x - lstsq| = {np.linalg.norm(x_lin - np.linalg.lstsq(A, b, rcond=None)[0]):.1e}")
assert k_lin == 1

(в) Ньютон: [-0.      3.8658 -6.4737] за 9 итераций, f = 64.42;  Гаусс–Ньютон: [2.1014 2.9967 0.7003] за 7, f = 1.1393
    ранг J в точке Ньютона: 3,  |grad f| там: 1.2e-09,  f в старте: 33.87 < f в точке Ньютона: 64.42
(г) линейный МНК: Гаусс–Ньютон за 1 итерацию, |x - lstsq| = 0.0e+00


## 11.8. Прочитайте график

> Ошибки трёх методов с $\lVert e_0\rVert=1$: (А) $0.98,\ 0.96,\ 0.94,\dots$; (Б) $0.3,\ 0.05,\ 3\cdot10^{-3},\ 10^{-5},\ 10^{-9}$; (В) $0.5,\ 0.1,\ 10^{-2},\ 10^{-4},\ 10^{-8},\ 10^{-16}$. Тип сходимости, метод из таблицы, число итераций до $10^{-12}$.
>
> *Суть: тип сходимости читается по отношениям соседних ошибок — повторение Л4, §6.*

**(А)** Отношения $0.98,\ 0.98,\ 0.98$ — постоянный множитель: **линейная** сходимость с $q=0.98$, то есть $\kappa\approx50$ при шаге $1/L$. Так ведёт себя градиентный спуск (или L-BFGS с маленькой памятью на плохо обусловленной задаче). До $10^{-12}$: $\ln10^{-12}/\ln0.98\approx1370$ итераций.

**(Б)** Отношения $0.17,\ 0.06,\ 0.003,\ 10^{-4}$ — убывают к нулю, но $\lVert e_{k+1}\rVert/\lVert e_k\rVert^2$ не постоянно ($0.6,\ 1.2,\ 1.1,\ 10$ — растёт): **сверхлинейная**, не квадратичная — почерк BFGS. Следующая ошибка порядка $10^{-9}\cdot10^{-5}=10^{-14}$: ещё одна итерация, всего $5$.

**(В)** Каждая ошибка — квадрат предыдущей ($0.5\to0.25$? нет: $0.1$; дальше $10^{-2},\ 10^{-4},\ 10^{-8},\ 10^{-16}$ — ровно квадраты): **квадратичная**, Ньютон или Гаусс–Ньютон при нулевых невязках. $10^{-12}$ достигается на пятой итерации ($10^{-16}$); первый шаг $0.5\to0.1$ — ещё «горб», модель только начала работать.

In [14]:
seqs = {"А": 0.98 ** np.arange(1, 6), "Б": np.array([0.3, 0.05, 3e-3, 1e-5, 1e-9]), "В": np.array([0.5, 0.1, 1e-2, 1e-4, 1e-8, 1e-16])}
for name, e in seqs.items():
    r1 = e[1:] / e[:-1]; r2 = e[1:] / e[:-1] ** 2
    kind = "линейная" if np.allclose(r1, r1[0], rtol=0.05) else ("квадратичная" if np.allclose(r2[1:], r2[1], rtol=0.5) else "сверхлинейная")
    print(f"({name}) e_(k+1)/e_k = {np.array2string(r1, precision=3)};  e_(k+1)/e_k^2 = {np.array2string(r2, precision=2)}  -> {kind}")
print("(А) итераций до 1e-12 при q = 0.98:", int(np.ceil(np.log(1e-12) / np.log(0.98))))

(А) e_(k+1)/e_k = [0.98 0.98 0.98 0.98];  e_(k+1)/e_k^2 = [1.   1.02 1.04 1.06]  -> линейная
(Б) e_(k+1)/e_k = [1.667e-01 6.000e-02 3.333e-03 1.000e-04];  e_(k+1)/e_k^2 = [ 0.56  1.2   1.11 10.  ]  -> сверхлинейная
(В) e_(k+1)/e_k = [2.e-01 1.e-01 1.e-02 1.e-04 1.e-08];  e_(k+1)/e_k^2 = [0.4 1.  1.  1.  1. ]  -> квадратичная
(А) итераций до 1e-12 при q = 0.98: 1368


## 11.9. Цепь: квази-Ньютон на квадратичной (с кодом)

> (а) `bfgs` с точным шагом (`Q=H`) на цепи $N=10,20,40,80$: сколько итераций и почему не больше числа различных собственных чисел $H$. (б) То же с backtracking: для каких $N$ сходится за $10\,000$ итераций, для каких нет; что сломалось. (в) `L-BFGS-B` с `maxcor=5` на $N=40,80,160,400$ против `np.linalg.solve`.
>
> *Суть: BFGS с точным шагом конечен на квадратичной, а без правильного линейного поиска хрупок.*

**(а)** $5,\ 10,\ 20,\ 40$ — ровно $N/2$. Теория: BFGS с точным линейным поиском на квадратичной функции порождает те же направления, что метод сопряжённых градиентов (лекция 6), а тот сходится не более чем за число *различных* собственных чисел матрицы. У $H=\operatorname{diag}(DK,DK)$ собственные числа $D(2-2\cos\frac{j\pi}{N+1})$, $j=1..N$, каждое дважды — различных ровно $N$. Почему хватает $N/2$: старт (прямая между концами) и решение (парабола) симметричны относительно середины цепи, поэтому начальная ошибка раскладывается только по симметричным собственным векторам — их $N/2$.

**(б)** С одним backtracking (шаг $\le1$) метод хрупок: при $N=10$ сходится за $42$ итерации, при $N=20$ и $N=40$ не сходится за $10\,000$, при $N=80$ снова сходится ($132$). Исход зависит от деталей округления, и это само по себе диагноз. Что ломается: в какой-то момент принятый шаг оказывается коротким, пара $(s,y)$ с крошечным $s$ почти не несёт информации о кривизне, но обновление всё равно меняет $H$; направление $p=-Hg$ становится слишком длинным, Армихо делит его десятки раз ($37$ вычислений $E$ на итерацию), принятые шаги падают до $10^{-9}$ — и метод стоит на месте, хотя $\lVert\nabla E\rVert$ уже близка к порогу. Лекарства: точный шаг (на квадратичной он есть), условия Вольфе — принимать и $\alpha>1$, когда кривизна мала (лекция 6), — или хотя бы масштабирование $H_0=\frac{s^\top y}{y^\top y}I$ после первого шага.

**(в)** L-BFGS с памятью 5 на $N=40,80,160,400$: $75,\ 144,\ 276,\ \approx660$ итераций — растёт примерно линейно по $N$; при $N=400$ это уже секунды (почти всё время — наши плотные умножения $Hv$ размера $800\times800$), а `np.linalg.solve` на той же матрице — миллисекунды. Заметьте и точность: при $N=400$ L-BFGS остановился по $\lVert\nabla E\rVert\le10^{-6}$, но ошибка решения — десятые доли: при $\kappa\approx65\,000$ малый градиент ещё не означает близости к минимуму (Л4, §4.6). На этой задаче Ньютон вне конкуренции; L-BFGS выигрывает там, где матрицу $n\times n$ нельзя ни собрать, ни хранить.

In [15]:
# (а): точный шаг — ровно N/2 итераций
for N in (10, 20, 40, 80):
    energy, grad_E, H, b, v0, _ = make_chain(N)
    _, k, _, _ = bfgs(energy, grad_E, v0, Q=H)
    n_distinct = len(np.unique(np.linalg.eigvalsh(H).round(8)))
    print(f"(а) N = {N:3d}: BFGS с точным шагом — {k} итераций; различных собственных чисел H: {n_distinct}")
    assert k == N // 2

(а) N =  10: BFGS с точным шагом — 5 итераций; различных собственных чисел H: 10
(а) N =  20: BFGS с точным шагом — 10 итераций; различных собственных чисел H: 20
(а) N =  40: BFGS с точным шагом — 20 итераций; различных собственных чисел H: 40
(а) N =  80: BFGS с точным шагом — 40 итераций; различных собственных чисел H: 80


In [16]:
# (б): backtracking — хрупок: исход зависит от N
status = {}
for N in (10, 20, 40, 80):
    energy, grad_E, H, b, v0, _ = make_chain(N)
    v, k, path, nfev = bfgs(energy, grad_E, v0, maxit=10_000)
    steps = np.linalg.norm(np.diff(path, axis=0), axis=1); status[N] = k < 10_000
    print(f"(б) N = {N:3d}: {'сошёлся за %5d' % k if k < 10_000 else 'НЕ сошёлся за 10000'} итераций, |grad E| = {np.linalg.norm(grad_E(v)):.1e}, "
          f"вычислений E на итерацию: {nfev / k:2.0f}, последние |s_k|: {np.array2string(steps[-2:], formatter={'float_kind': '{:.1e}'.format})}")
assert any(status.values()) and not all(status.values()), status

(б) N =  10: сошёлся за    42 итераций, |grad E| = 1.0e-06, вычислений E на итерацию:  7, последние |s_k|: [7.9e-09 7.8e-09]


(б) N =  20: НЕ сошёлся за 10000 итераций, |grad E| = 4.1e-06, вычислений E на итерацию: 37, последние |s_k|: [0.0e+00 0.0e+00]


(б) N =  40: НЕ сошёлся за 10000 итераций, |grad E| = 3.5e-06, вычислений E на итерацию: 37, последние |s_k|: [0.0e+00 0.0e+00]
(б) N =  80: сошёлся за   132 итераций, |grad E| = 2.7e-07, вычислений E на итерацию:  7, последние |s_k|: [1.9e-07 1.4e-08]


In [17]:
# (в): L-BFGS против solve
import time
print(" N     L-BFGS m=5 итераций   время L-BFGS   время solve")
for N in (40, 80, 160, 400):
    energy, grad_E, H, b, v0, _ = make_chain(N)
    t0 = time.perf_counter(); r = minimize(energy, v0, jac=grad_E, method="L-BFGS-B", options={"gtol": 1e-6, "maxiter": 100_000, "maxcor": 5}); t_l = time.perf_counter() - t0
    t0 = time.perf_counter(); v = np.linalg.solve(H, -b); t_s = time.perf_counter() - t0
    print(f"{N:3d}   {r.nit:12d}            {t_l:7.2f} с      {t_s * 1e3:6.1f} мс     |v - v*| = {np.linalg.norm(r.x - v):.1e}  (κ = {np.linalg.cond(H):.0f})")

 N     L-BFGS m=5 итераций   время L-BFGS   время solve
 40             75               0.09 с         0.2 мс     |v - v*| = 2.4e-04  (κ = 681)


 80            144               0.01 с       112.0 мс     |v - v*| = 5.5e-03  (κ = 2658)


160            276               0.13 с        97.6 мс     |v - v*| = 3.2e-02  (κ = 10505)


400            663               8.75 с       101.1 мс     |v - v*| = 3.7e-01  (κ = 65170)


## 11.10. Демпфированный Ньютон из разных стартов (с кодом)

> (а) Чистый и демпфированный Ньютон на Розенброке из 10 случайных стартов в $[-2,2]\times[-1,3]$ (`default_rng(2026)`): итерации и наибольшая ошибка по пути. (б) То же на Химмельблау: куда приходит каждый; меняет ли демпфирование долю минимумов.
>
> *Суть: линейный поиск убирает выбросы, но не меняет предел.*

**(а)** Чистый Ньютон на Розенброке сходится из всех десяти стартов за $2$–$8$ итераций, но по пути ошибка подскакивает (до нескольких единиц — выбросы вроде $(0.76,-3.18)$). Демпфированный делает больше итераций ($10$–$25$), зато $f$ убывает монотонно и наибольшая ошибка по пути не превышает стартовой.

**(б)** На Химмельблау чистый Ньютон приходит в сёдла и максимум примерно из трети стартов (конспект, §3.3). Демпфированный Ньютон с регуляризацией $H+\tau I$ при индефинитном гессиане *всегда* спускается, поэтому в максимум он прийти не может, и доля минимумов растёт — но предел по-прежнему локальный: какой из четырёх минимумов, зависит от старта. Линейный поиск гарантирует спуск, а не глобальность.

In [18]:
rng = np.random.default_rng(2026)
starts10 = np.c_[rng.uniform(-2, 2, 10), rng.uniform(-1, 3, 10)]
print("(а) Розенброк: старт -> итераций (чистый / демпфированный), наибольшая ошибка по пути (чистый / демпфированный)")
for s in starts10:
    _, kp, pp = newton(rosen_grad, rosen_hess, s); _, kd, pd = newton_damped(rosen, rosen_grad, rosen_hess, s)
    print(f"    {s.round(2)} -> {kp:2d} / {kd:2d},   {np.linalg.norm(pp - 1, axis=1).max():5.2f} / {np.linalg.norm(pd - 1, axis=1).max():5.2f}")

(а) Розенброк: старт -> итераций (чистый / демпфированный), наибольшая ошибка по пути (чистый / демпфированный)
    [-1.28  2.87] ->  6 / 23,    5.16 /  2.95
    [0.56 2.68] ->  5 / 11,    1.74 /  1.74
    [-0.13  1.54] ->  5 / 15,    1.50 /  1.50
    [-0.52  2.01] ->  5 / 19,    2.31 /  1.82
    [-0.58  1.06] ->  6 / 18,    2.49 /  1.72
    [1.16 2.3 ] ->  5 /  8,    1.31 /  1.31
    [1.62 0.79] ->  5 / 13,    1.73 /  1.73
    [-1.29  0.36] ->  6 / 23,    5.12 /  2.38
    [0.61 0.11] ->  5 / 10,    0.97 /  0.97
    [-0.81 -0.09] ->  6 / 20,    3.14 /  2.11


In [19]:
himm = lambda x: (x[0] ** 2 + x[1] - 11) ** 2 + (x[0] + x[1] ** 2 - 7) ** 2
himm_grad = lambda x: np.array([4 * x[0] * (x[0] ** 2 + x[1] - 11) + 2 * (x[0] + x[1] ** 2 - 7), 2 * (x[0] ** 2 + x[1] - 11) + 4 * x[1] * (x[0] + x[1] ** 2 - 7)])
himm_hess = lambda x: np.array([[12 * x[0] ** 2 + 4 * x[1] - 42, 4 * x[0] + 4 * x[1]], [4 * x[0] + 4 * x[1], 4 * x[0] + 12 * x[1] ** 2 - 26]])
kind = lambda x: ("минимум" if np.linalg.eigvalsh(himm_hess(x)).min() > 0 else ("максимум" if np.linalg.eigvalsh(himm_hess(x)).max() < 0 else "седло"))
print("(б) Химмельблау: старт -> чистый Ньютон | демпфированный")
n_min = [0, 0]
for s in starts10 * 1.5:                                            # растягиваем старты на [-3,3] x [-1.5,4.5]
    xp, kp, _ = newton(himm_grad, himm_hess, s); xd, kd, _ = newton_damped(himm, himm_grad, himm_hess, s)
    n_min[0] += kind(xp) == "минимум"; n_min[1] += kind(xd) == "минимум"
    print(f"    {s.round(2)} -> {xp.round(3)} {kind(xp):8s} за {kp:2d} | {xd.round(3)} {kind(xd):8s} за {kd:2d}")
print(f"    минимумов: чистый {n_min[0]}/10, демпфированный {n_min[1]}/10")
assert n_min[1] >= n_min[0]

(б) Химмельблау: старт -> чистый Ньютон | демпфированный
    [-1.93  4.3 ] -> [-2.805  3.131] минимум  за  6 | [-2.805  3.131] минимум  за  6
    [0.84 4.02] -> [0.087 2.884] седло    за  5 | [0.087 2.884] седло    за  5
    [-0.2   2.32] -> [0.087 2.884] седло    за  5 | [0.087 2.884] седло    за  5
    [-0.78  3.02] -> [0.087 2.884] седло    за  4 | [-2.805  3.131] минимум  за  6
    [-0.87  1.59] -> [-2.805  3.131] минимум  за 12 | [-2.805  3.131] минимум  за  7
    [1.74 3.46] -> [3. 2.] минимум  за 11 | [3. 2.] минимум  за  7
    [2.43 1.19] -> [3.385 0.074] седло    за  6 | [3. 2.] минимум  за  5
    [-1.94  0.53] -> [-3.073 -0.081] седло    за  9 | [-3.073 -0.081] седло    за  6
    [0.92 0.17] -> [-0.271 -0.923] максимум за  5 | [3.385 0.074] седло    за  6
    [-1.21 -0.14] -> [-0.271 -0.923] максимум за  5 | [-3.779 -3.283] минимум  за  5
    минимумов: чистый 3/10, демпфированный 6/10
